# Kannada NER: Unigram LM + Bi-LSTM
This notebook trains a Named Entity Recognition model for Kannada using **Unigram Language Model** tokenization (SentencePiece/T5 style).

### **Standard Features Implemented**:
1. **Dataset Splitting**: Train (200k), Validation (10k), Test (10k).
2. **Early Stopping**: Prevents overfitting by monitoring Validation Loss.
3. **Evaluation Loop**: Computes Accuracy on Val/Test sets.
4. **Metaspace Tokenization**: Handles whitespace-agnostic subword discovery.

In [ ]:
# 1. Installation
!pip install datasets==2.16.0 tokenizers torch tqdm scikit-learn

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
    Fou

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import json
import os
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on: {DEVICE}")

Executing on: cuda


In [ ]:
VOCAB_SIZE = 10000
CORPUS_SAMPLES = 60000

TRAIN_SAMPLES = 200000
VAL_SAMPLES = 10000
TEST_SAMPLES = 10000
PATIENCE = 3

def train_tokenizer():
    print("Streaming data for Unigram Tokenizer training...")
    # Note: HF_TOKEN warning in Colab is normal and can be ignored for public datasets.
    raw_ds = load_dataset("ai4bharat/naamapadam", "kn", split="train", streaming=True, trust_remote_code=True)

    def corpus_iterator():
        for i, row in enumerate(raw_ds.take(CORPUS_SAMPLES)):
            yield " ".join(row['tokens'])

    # Initialize Unigram Model
    tokenizer = Tokenizer(models.Unigram())
    # Metaspace pre-tokenization (SentencePiece style)
    # Using no arguments for maximum compatibility across tokenizer versions
    tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

    trainer = trainers.UnigramTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
        unk_token="[UNK]"
    )

    tokenizer.train_from_iterator(corpus_iterator(), trainer=trainer)
    tokenizer.save("kannada_unigram.json")
    return tokenizer

tokenizer = train_tokenizer()

Streaming data for Unigram Tokenizer training...


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
class Unigram_NERModel(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=128, hidden_dim=256):
        super(Unigram_NERModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embeds = self.dropout(self.embeddings(x))
        lstm_out, _ = self.lstm(embeds)
        return self.fc(self.dropout(lstm_out))

In [ ]:
class NER_Unigram_Dataset(IterableDataset):
    def __init__(self, ds, tokenizer, max_len=128):
        self.ds = ds
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __iter__(self):
        for row in self.ds:
            tokens = row['tokens']
            ner_tags = row['ner_tags']

            subword_ids = []
            label_ids = []

            for word, tag in zip(tokens, ner_tags):
                encoded = self.tokenizer.encode(" " + word, add_special_tokens=False).ids
                subword_ids.extend(encoded)
                label_ids.extend([tag] * len(encoded))

            subword_ids = subword_ids[:self.max_len]
            label_ids = label_ids[:self.max_len]
            yield torch.tensor(subword_ids), torch.tensor(label_ids)

def collate_fn(batch):
    x, y = zip(*batch)
    x_padded = pad_sequence(x, batch_first=True, padding_value=0)
    y_padded = pad_sequence(y, batch_first=True, padding_value=-1)
    return x_padded, y_padded

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    batch_count = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            batch_count += 1
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=2).view(-1).cpu().numpy()
            targ = y.view(-1).cpu().numpy()
            mask = targ != -1
            all_preds.extend(preds[mask])
            all_targets.extend(targ[mask])

    avg_loss = total_loss / max(1, batch_count)
    accuracy = accuracy_score(all_targets, all_preds) if all_targets else 0
    return avg_loss, accuracy

In [ ]:
TAG_MAP = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4, "B-LOC": 5, "I-LOC": 6}
model = Unigram_NERModel(tokenizer.get_vocab_size(), len(TAG_MAP)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(ignore_index=-1)

full_ds = load_dataset("ai4bharat/naamapadam", "kn", split="train", streaming=True, trust_remote_code=True)
ds_train = full_ds.take(TRAIN_SAMPLES)
ds_val = full_ds.skip(TRAIN_SAMPLES).take(VAL_SAMPLES)
ds_test = full_ds.skip(TRAIN_SAMPLES + VAL_SAMPLES).take(TEST_SAMPLES)

train_loader = DataLoader(NER_Unigram_Dataset(ds_train, tokenizer), batch_size=64, collate_fn=collate_fn)
val_loader = DataLoader(NER_Unigram_Dataset(ds_val, tokenizer), batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(NER_Unigram_Dataset(ds_test, tokenizer), batch_size=64, collate_fn=collate_fn)

print("Starting Unigram NER Training with Early Stopping...")
best_val_loss = float('inf')
no_improve = 0
epoch = 0

while True:
    epoch += 1
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, len(TAG_MAP)), y.view(-1))
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())

    v_loss, v_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch}: Val Loss {v_loss:.4f}, Val Acc {v_acc:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        no_improve = 0
        torch.save(model.state_dict(), "kannada_ner_unigram_v3.pth")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early Stopping Triggered!")
            break

Starting Unigram NER Training with Early Stopping...


Epoch 1: 0it [00:00, ?it/s]

Epoch 1: Val Loss 0.3606, Val Acc 0.8840


Epoch 2: 0it [00:00, ?it/s]

Epoch 2: Val Loss 0.3149, Val Acc 0.8988


...

Epoch 6: Val Loss 0.2972, Val Acc 0.9041


Epoch 7: 0it [00:00, ?it/s]

Epoch 7: Val Loss 0.2973, Val Acc 0.9044


Epoch 8: 0it [00:00, ?it/s]

Epoch 8: Val Loss 0.2969, Val Acc 0.9041
Early Stopping Triggered!


## 6. Final Test Accuracy Block

In [ ]:
model.load_state_dict(torch.load("kannada_ner_unigram_v3.pth"))
t_loss, t_acc = evaluate(model, test_loader, criterion)
print(f"\nFINAL TEST RESULTS (Unigram Model):")
print(f"Test Loss: {t_loss:.4f}")
print(f"Test Accuracy: {t_acc:.4f}")


FINAL TEST RESULTS (Unigram Model):
Test Loss: 0.2934
Test Accuracy: 0.9047


In [ ]:
def predict(text):
    model.eval()
    inv_map = {v: k for k, v in TAG_MAP.items()}
    tokens = text.split()
    ids = []
    token_to_subwords = {}

    for t in tokens:
        sub_tkns = tokenizer.encode(" "+t, add_special_tokens=False).tokens
        token_to_subwords[t] = sub_tkns
        ids.extend(tokenizer.encode(" "+t, add_special_tokens=False).ids)

    input_t = torch.tensor([ids]).to(DEVICE)
    with torch.no_grad():
        logits = model(input_t)
        preds = torch.argmax(logits, dim=2)[0].cpu().numpy()

    print(f"Original: {text}\n")
    idx = 0
    for t in tokens:
        sub_tokens = token_to_subwords[t]
        tag = inv_map[preds[idx]]
        print(f"{t:15} -> {tag:6} (Subwords: {sub_tokens})")
        idx += len(sub_tokens)



In [ ]:
predict("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")

Original: ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.

ಬೆಂಗಳೂರಿನ       -> B-LOC  (Subwords: ['▁ಬೆಂಗಳೂರಿನ'])
ಇಂದಿರಾ          -> O      (Subwords: ['▁ಇಂದಿರಾ'])
ಅವರ             -> O      (Subwords: ['▁ಅವರ'])
ತಾಯಿ            -> O      (Subwords: ['▁ತಾಯಿ'])
ಸುಧಾ            -> B-PER  (Subwords: ['▁ಸುಧಾ'])
ಅವರ             -> O      (Subwords: ['▁ಅವರ'])
ಮನೆಗೆ           -> O      (Subwords: ['▁ಮನೆ', 'ಗೆ'])
ನುಗ್ಗಿ          -> O      (Subwords: ['▁ನುಗ್ಗ', 'ಿ'])
ಕಳ್ಳರು          -> O      (Subwords: ['▁ಕಳ್ಳ', 'ರು'])
ದರೋಡೆ           -> O      (Subwords: ['▁ದರೋಡೆ'])
ಮಾಡಿದ್ದಾರೆ.     -> O      (Subwords: ['▁ಮಾಡಿದ್ದಾರೆ', '.'])
ಮಾರತಹಳ್ಳಿ       -> B-LOC  (Subwords: ['▁ಮಾ', 'ರ', 'ತ', 'ಹಳ್ಳಿ'])
ನಿವಾಸಿ          -> O      (Subwords: ['▁ನಿವಾಸಿ'])
ಮಹೇಶ್           -> B-PER  (Subwords: ['▁ಮಹೇಶ್'])
ಅವರು            -> O      (Subwords: ['▁ಅವರು'])
ಪೊಲೀಸ್          -> O      (Subwords: ['▁ಪೊಲೀಸ್'])
ಠಾಣೆಗೆ          -> O      (Subwords:

In [ ]:
predict("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")


Original: ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.

ಬೆಂಗಳೂರಿನ       -> B-LOC  (Subwords: ['▁ಬೆಂಗಳೂರಿನ'])
ಇಂದಿರಾನಗರದಲ್ಲಿ  -> B-LOC  (Subwords: ['▁ಇಂದಿರಾ', 'ನಗರ', 'ದಲ್ಲಿ'])
ಸುಧಾ            -> O      (Subwords: ['▁ಸುಧಾ'])
ಅವರ             -> O      (Subwords: ['▁ಅವರ'])
ಮನೆಗೆ           -> O      (Subwords: ['▁ಮನೆ', 'ಗೆ'])
ನುಗ್ಗಿ          -> O      (Subwords: ['▁ನುಗ್ಗ', 'ಿ'])
ಕಳ್ಳರು          -> O      (Subwords: ['▁ಕಳ್ಳ', 'ರು'])
ದರೋಡೆ           -> O      (Subwords: ['▁ದರೋಡೆ'])
ಮಾಡಿದ್ದಾರೆ.     -> O      (Subwords: ['▁ಮಾಡಿದ್ದಾರೆ', '.'])
ನಿವಾಸಿ          -> O      (Subwords: ['▁ನಿವಾಸಿ'])
ಮಹೇಶ್           -> B-PER  (Subwords: ['▁ಮಹೇಶ್'])
ಅವರು            -> O      (Subwords: ['▁ಅವರು'])
ಪೊಲೀಸ್          -> O      (Subwords: ['▁ಪೊಲೀಸ್'])
ಠಾಣೆಗೆ          -> O      (Subwords: ['▁ಠಾಣೆ', 'ಗೆ'])
ದೂರು            -> O      (Subwords: ['▁ದೂರು'])
ನೀಡಿದರು.        -> O      (Subwords: ['▁ನೀಡಿದ', 'ರು', '.'])


In [ ]:
predict("ಬೆಸ್ಕಾಂ ಅಧಿಕಾರಿಗಳು ಮೂರು ದಿನಗಳಿಂದ ವಿದ್ಯುತ್ ಸಮಸ್ಯೆ ಬಗೆಹರಿಸಿಲ್ಲ. ಗ್ರಾಹಕ ಶಂಕರ್ ಅವರು ದೂರು ನೀಡಿದರು.")

Original: ಬೆಸ್ಕಾಂ ಅಧಿಕಾರಿಗಳು ಮೂರು ದಿನಗಳಿಂದ ವಿದ್ಯುತ್ ಸಮಸ್ಯೆ ಬಗೆಹರಿಸಿಲ್ಲ. ಗ್ರಾಹಕ ಶಂಕರ್ ಅವರು ದೂರು ನೀಡಿದರು.

ಬೆಸ್ಕಾಂ         -> O      (Subwords: ['▁ಬೆ', 'ಸ್ಕಾ', 'ಂ'])
ಅಧಿಕಾರಿಗಳು      -> O      (Subwords: ['▁ಅಧಿಕಾರಿಗಳು'])
ಮೂರು            -> O      (Subwords: ['▁ಮೂರು'])
ದಿನಗಳಿಂದ        -> O      (Subwords: ['▁ದಿನ', 'ಗಳಿಂದ'])
ವಿದ್ಯುತ್        -> O      (Subwords: ['▁ವಿದ್ಯುತ್'])
ಸಮಸ್ಯೆ          -> O      (Subwords: ['▁ಸಮಸ್ಯೆ'])
ಬಗೆಹರಿಸಿಲ್ಲ.    -> O      (Subwords: ['▁ಬಗೆ', 'ಹ', 'ರಿಸ', 'ಿಲ್ಲ', '.'])
ಗ್ರಾಹಕ          -> O      (Subwords: ['▁ಗ್ರಾಹಕ'])
ಶಂಕರ್           -> B-PER  (Subwords: ['▁ಶಂಕರ್'])
ಅವರು            -> O      (Subwords: ['▁ಅವರು'])
ದೂರು            -> O      (Subwords: ['▁ದೂರು'])
ನೀಡಿದರು.        -> O      (Subwords: ['▁ನೀಡಿದ', 'ರು', '.'])
